# Stage 5: Evaluation Metrics

This notebook demonstrates the full evaluation framework on experiment results:
1. **Verdict accuracy** — per-class precision/recall/F1, macro-F1, confusion matrix
2. **Bootstrap confidence intervals** — 95% CIs on accuracy
3. **McNemar's test** — statistical significance between experiment pairs
4. **LLM-as-Judge** — explanation quality scoring (faithfulness, specificity, completeness, nuance)
5. **Grounding rate** — % of factual statements traceable to evidence
6. **Pairwise comparison** — head-to-head explanation quality

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 5.1 Load experiment results

Load results saved by Stage 4 (notebook 04). If not available, run a quick experiment here.

In [ ]:
results_dir = Path("results/experiments")

experiments = {}
for path in sorted(results_dir.glob("E*.json")):
    with open(path) as f:
        data = json.load(f)
    experiments[data["experiment_id"]] = data
    print(f"{data['experiment_id']}: {data['config']['name']} — {data['correct']}/{data['total_claims']} ({data['accuracy']:.1%})")

if not experiments:
    print("No experiment results found. Run notebook 04 first, or run experiments below.")

In [ ]:
# If no saved results, run E1 and E2 as a quick demo
if not experiments:
    from src.pipelines.configurable import run_experiment
    from src.experiment_runner import EXPERIMENT_CONFIGS
    
    with open("data/test_claims.json") as f:
        claims = json.load(f)
    
    for exp_id in ["E1", "E2"]:
        config = EXPERIMENT_CONFIGS[exp_id]
        pipeline_config = {k: v for k, v in config.items() if k != "name"}
        print(f"Running {exp_id}...")
        results = []
        for c in claims:
            r = run_experiment(c["claim"], **pipeline_config)
            r["expected_verdict"] = c["expected_verdict"]
            r["correct"] = r["verdict"] == c["expected_verdict"]
            results.append(r)
        correct = sum(1 for r in results if r["correct"])
        experiments[exp_id] = {"experiment_id": exp_id, "config": config,
                               "total_claims": len(results), "correct": correct,
                               "accuracy": correct/len(results), "results": results}
        print(f"  {exp_id}: {correct}/{len(results)}")

## 5.2 Verdict accuracy — Macro-F1 and per-class metrics

In [ ]:
from src.evaluation.metrics import compute_verdict_accuracy, VERDICT_CLASSES

accuracy_results = {}
for exp_id, data in experiments.items():
    results = data["results"]
    expected = [{"expected_verdict": r["expected_verdict"]} for r in results]
    acc = compute_verdict_accuracy(results, expected)
    accuracy_results[exp_id] = acc

# Summary table
acc_table = pd.DataFrame([
    {
        "Experiment": exp_id,
        "Accuracy": f"{acc['correct']}/{acc['total']}",
        "Accuracy %": f"{acc['accuracy']:.1%}",
        "Macro-F1": f"{acc['macro_f1']:.4f}",
    }
    for exp_id, acc in accuracy_results.items()
])
print("Verdict Accuracy Summary:")
acc_table

In [ ]:
# Per-class breakdown for each experiment
for exp_id, acc in accuracy_results.items():
    print(f"\n{exp_id} — Per-class metrics:")
    for cls in VERDICT_CLASSES:
        m = acc["per_class"][cls]
        print(f"  {cls:>25}: P={m['precision']:.2f} R={m['recall']:.2f} F1={m['f1']:.2f} (support={m['support']})")

## 5.3 Confusion matrix

In [ ]:
# Show confusion matrix for the first two experiments side by side
exp_ids = list(experiments.keys())[:2]
fig, axes = plt.subplots(1, len(exp_ids), figsize=(7 * len(exp_ids), 5))
if len(exp_ids) == 1:
    axes = [axes]

short_labels = ["SUP", "UNS", "OVR", "INS"]

for ax, exp_id in zip(axes, exp_ids):
    cm = accuracy_results[exp_id]["confusion_matrix"]
    matrix = [[cm[a][p] for p in VERDICT_CLASSES] for a in VERDICT_CLASSES]
    
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=short_labels, yticklabels=short_labels, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"{exp_id}: Confusion Matrix")

plt.tight_layout()
plt.show()

## 5.4 Bootstrap confidence intervals

In [ ]:
from src.evaluation.metrics import bootstrap_ci

ci_results = {}
for exp_id, data in experiments.items():
    scores = [1.0 if r["correct"] else 0.0 for r in data["results"]]
    ci = bootstrap_ci(scores)
    ci_results[exp_id] = ci
    print(f"{exp_id}: {ci['mean']:.1%} (95% CI: [{ci['lower']:.1%}, {ci['upper']:.1%}])")

# Plot with error bars
fig, ax = plt.subplots(figsize=(10, 5))
exp_ids = list(ci_results.keys())
means = [ci_results[e]["mean"] * 100 for e in exp_ids]
lowers = [ci_results[e]["lower"] * 100 for e in exp_ids]
uppers = [ci_results[e]["upper"] * 100 for e in exp_ids]
errors = [[m - l for m, l in zip(means, lowers)], [u - m for m, u in zip(means, uppers)]]

ax.bar(exp_ids, means, yerr=errors, capsize=8, color=["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"][:len(exp_ids)], alpha=0.8)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Verdict Accuracy with 95% Bootstrap Confidence Intervals")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 5.5 McNemar's test — pairwise statistical significance

In [ ]:
from src.evaluation.metrics import mcnemar_test

exp_ids = list(experiments.keys())
if len(exp_ids) >= 2:
    print(f"{'Pair':<15} {'Statistic':>10} {'p-value':>10} {'Significant':>12}")
    print("-" * 50)
    
    for i in range(len(exp_ids)):
        for j in range(i + 1, len(exp_ids)):
            a_id, b_id = exp_ids[i], exp_ids[j]
            a_results = [r["verdict"] for r in experiments[a_id]["results"]]
            b_results = [r["verdict"] for r in experiments[b_id]["results"]]
            expected = [r["expected_verdict"] for r in experiments[a_id]["results"]]
            
            result = mcnemar_test(a_results, b_results, expected)
            sig = "YES *" if result["significant"] else "no"
            print(f"{a_id} vs {b_id:<8} {result['statistic']:>10.4f} {result['p_value']:>10.6f} {sig:>12}")
else:
    print("Need at least 2 experiments for McNemar's test")

## 5.6 LLM-as-Judge explanation quality

Score explanations on 4 dimensions (1-5): faithfulness, specificity, completeness, nuance.

**Note:** This makes LLM API calls and costs money. Run selectively.

In [ ]:
from src.evaluation.llm_judge import score_explanation

# Score the first 2 experiments (or whichever you want to compare)
judge_exps = list(experiments.keys())[:2]
judge_scores = {}

for exp_id in judge_exps:
    print(f"\nScoring {exp_id} explanations...")
    scores = []
    for r in experiments[exp_id]["results"]:
        if r.get("verdict") == "ERROR":
            continue
        s = score_explanation(r["claim"], r["verdict"], r["explanation"], r["evidence"])
        scores.append(s)
        dims = {d: s[d]["score"] for d in ["faithfulness", "specificity", "completeness", "nuance"]}
        print(f"  {r['claim'][:45]:<47} F={dims['faithfulness']} S={dims['specificity']} C={dims['completeness']} N={dims['nuance']}")
    judge_scores[exp_id] = scores

print("\nDone!")

In [ ]:
# Compare average scores
dimensions = ["faithfulness", "specificity", "completeness", "nuance"]

avg_scores = {}
for exp_id in judge_exps:
    avgs = {}
    for dim in dimensions:
        dim_scores = [s[dim]["score"] for s in judge_scores[exp_id]]
        avgs[dim] = round(sum(dim_scores) / len(dim_scores), 2)
    avgs["overall"] = round(sum(avgs.values()) / 4, 2)
    avg_scores[exp_id] = avgs

score_df = pd.DataFrame(avg_scores).T
score_df.index.name = "Experiment"
print("Average Explanation Quality Scores (1-5):")
score_df

In [ ]:
# Radar chart / grouped bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(dimensions))
width = 0.35

for i, exp_id in enumerate(judge_exps):
    values = [avg_scores[exp_id][d] for d in dimensions]
    offset = (i - len(judge_exps)/2 + 0.5) * width
    bars = ax.bar(x + offset, values, width, label=f"{exp_id}: {experiments[exp_id]['config']['name']}")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f"{val}", ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([d.capitalize() for d in dimensions])
ax.set_ylabel("Score (1-5)")
ax.set_ylim(0, 5.5)
ax.set_title("Explanation Quality: LLM-as-Judge Scores")
ax.legend()
plt.tight_layout()
plt.show()

## 5.7 Grounding rate

In [ ]:
from src.evaluation.grounding_rate import compute_grounding_rate

grounding_results = {}
for exp_id in judge_exps:
    print(f"\nComputing grounding for {exp_id}...")
    rates = []
    for r in experiments[exp_id]["results"]:
        if r.get("verdict") == "ERROR":
            continue
        gr = compute_grounding_rate(r["explanation"], r["evidence"])
        rate = gr.get("grounding_rate", 0)
        rates.append(rate)
        print(f"  {r['claim'][:45]:<47} {gr.get('grounded_count', 0)}/{gr.get('total_statements', 0)} = {rate:.0%}")
    
    avg_rate = sum(rates) / len(rates) if rates else 0
    grounding_results[exp_id] = {"rates": rates, "avg_rate": avg_rate}
    print(f"  Average grounding rate: {avg_rate:.1%}")

In [ ]:
# Grounding rate comparison
if grounding_results:
    fig, ax = plt.subplots(figsize=(8, 5))
    exp_labels = list(grounding_results.keys())
    avg_rates = [grounding_results[e]["avg_rate"] * 100 for e in exp_labels]
    
    ax.bar(exp_labels, avg_rates, color=["#4C72B0", "#DD8452"][:len(exp_labels)])
    ax.set_ylabel("Grounding Rate (%)")
    ax.set_title("Average Grounding Rate by Experiment")
    ax.set_ylim(0, 100)
    for i, v in enumerate(avg_rates):
        ax.text(i, v + 1, f"{v:.0f}%", ha="center", fontweight="bold")
    plt.tight_layout()
    plt.show()

## 5.8 Pairwise explanation comparison

In [ ]:
from src.evaluation.pairwise import compare_explanations, compute_win_rates

if len(judge_exps) >= 2:
    exp_a, exp_b = judge_exps[0], judge_exps[1]
    print(f"Pairwise comparison: {exp_a} (A) vs {exp_b} (B)")
    print("=" * 60)
    
    comparisons = []
    results_a = experiments[exp_a]["results"]
    results_b = experiments[exp_b]["results"]
    
    for ra, rb in zip(results_a, results_b):
        if ra.get("verdict") == "ERROR" or rb.get("verdict") == "ERROR":
            continue
        comp = compare_explanations(
            ra["claim"], ra["explanation"], rb["explanation"],
            ra["evidence"], rb["evidence"]
        )
        comparisons.append(comp)
        winner = comp.get("overall", {}).get("winner", "?")
        print(f"  {ra['claim'][:45]:<47} Winner: {winner}")
    
    win_rates = compute_win_rates(comparisons)
    print(f"\nWin rates: {exp_a}={win_rates['a_win_rate']:.0%}  {exp_b}={win_rates['b_win_rate']:.0%}  Ties={win_rates['ties']}/{win_rates['total']}")
else:
    print("Need at least 2 experiments for pairwise comparison")

## 5.9 Final summary dashboard

In [ ]:
final_summary = []
for exp_id in experiments:
    row = {
        "Experiment": exp_id,
        "Name": experiments[exp_id]["config"]["name"][:30],
        "Accuracy": f"{experiments[exp_id]['accuracy']:.1%}",
        "Macro-F1": accuracy_results.get(exp_id, {}).get("macro_f1", "N/A"),
    }
    if exp_id in ci_results:
        ci = ci_results[exp_id]
        row["95% CI"] = f"[{ci['lower']:.1%}, {ci['upper']:.1%}]"
    if exp_id in avg_scores:
        row["Explanation Quality"] = avg_scores[exp_id]["overall"]
    if exp_id in grounding_results:
        row["Grounding Rate"] = f"{grounding_results[exp_id]['avg_rate']:.0%}"
    final_summary.append(row)

final_df = pd.DataFrame(final_summary)
print("\nFinal Evaluation Summary:")
final_df

## Key Takeaways

**Metrics implemented:**
- Verdict accuracy with Macro-F1 and per-class breakdown
- Bootstrap 95% confidence intervals (accounts for small sample uncertainty)
- McNemar's test (tells you if differences are statistically significant)
- LLM-as-Judge explanation quality (4 dimensions, no reference explanations needed)
- Grounding rate (% hallucination detection)
- Pairwise head-to-head comparison

**For the full evaluation with 120+ claims:** Run `python -m src.experiment_runner E1` through E7, then feed the results into this notebook.